# RAG Retrieval (FAISS)

This notebook handles the retrieval phase of the RAG evaluation pipeline using a local FAISS index.
It is the FAISS equivalent of `rag_retrieval.ipynb` and produces identical output files,
so the eval notebooks can consume the results without any changes.

## Pipeline
1. **Load** the QA dataset (questions + ground-truth Wikipedia IDs)
2. **Load** the FAISS index from disk (memory-mapped for low RAM usage)
3. **Retrieve** top-K documents for every question using each strategy
4. **Save** raw retrieval results as Parquet files

## Strategies
- **IVF-PQ** (`ivfpq`): Approximate nearest-neighbour search via FAISS IVF+PQ index

## Output
Results are saved to `{RESULTS_DIR}/` as `results_{strategy}.parquet`.  
These files are consumed by the eval notebooks for metric computation and visualization.

> **Note:** Run this notebook once (or when you change retrieval settings).  
> The evaluation notebooks can be re-run cheaply on saved results.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))

## 1. Configuration

In [2]:
import json
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

from config import DATA_DIR
from src.rag.faiss_rag_service import FaissRagService, MemoryConfig
from src.rag.utils import IndexingConfig
from src.metrics.decile_utils import COL_DECILE_UNWEIGHTED, COL_DECILE_CHUNK_WEIGHTED

warnings.filterwarnings("ignore")
load_dotenv()

# ── FAISS Index ───────────────────────────────────────────────────────────────
FAISS_INDEX_DIR = DATA_DIR / "wiki_full_bil" / "faiss_high"   # Root dir containing faiss/index.faiss
NAME  = "ivfpq_high"                        # Must match how the index was built
FAISS_STRATEGY = "ivfpq"              # Name for results (must match strategy name in eval)
IVFPQ_NPROBE    = 512                            # Cells to probe at query time (recall vs speed)
USE_MMAP        = True                           # Memory-map for low RAM usage

# ── QA Dataset ───────────────────────────────────────────────────────────────
COLLECTION_NAME  = "wiki_full_bil"
COLLECTION_ROOT  = DATA_DIR / COLLECTION_NAME
OUTPUT_NAME      = "all_qa_8k"
QUESTIONS_PATH   = COLLECTION_ROOT / f"{OUTPUT_NAME}.parquet"

# ── Embedding (MUST match what the index was built with) ──────────────────────
EMBEDDING_PROVIDER    = "modal"
EMBEDDING_MODEL       = "Lajavaness/bilingual-embedding-small"
CHUNK_SIZE            = 1000
CHUNK_OVERLAP         = 100
GPU_BATCH_SIZE        = 512
REQUEST_BATCH_SIZE    = 512
NORMALISE_EMBEDDINGS  = True

# ── Retrieval ─────────────────────────────────────────────────────────────────
STRATEGIES        = [FAISS_STRATEGY]   # Strategies to evaluate
TOP_K             = 10                # Primary metric top-K (matches ES retrieval notebook)
K_VALUES_DETAILED = [1, 3, 5, 10]     # K values for investigation/curves
MAX_QUESTIONS     = None               # Set to int to limit for testing

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_FOLDER = "all_qa_8k"        # Folder name — must match ES results folder for joint eval
RESULTS_DIR   = COLLECTION_ROOT / OUTPUT_FOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  FAISS index:    {FAISS_INDEX_DIR}")
print(f"  FAISS strategy: {FAISS_STRATEGY}  (nprobe={IVFPQ_NPROBE}, mmap={USE_MMAP})")
print(f"  Questions:      {QUESTIONS_PATH}")
print(f"  Embedding:      {EMBEDDING_MODEL}")
print(f"  Strategies:     {STRATEGIES}")
print(f"  Top-K:          {TOP_K}")
print(f"  K values:       {K_VALUES_DETAILED}")
print(f"  Results dir:    {RESULTS_DIR}")

Configuration:
  FAISS index:    /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/faiss_high
  FAISS strategy: ivfpq  (nprobe=512, mmap=True)
  Questions:      /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k.parquet
  Embedding:      Lajavaness/bilingual-embedding-small
  Strategies:     ['ivfpq']
  Top-K:          10
  K values:       [1, 3, 5, 10]
  Results dir:    /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k


## 2. Load Questions

Load the QA dataset containing questions, ground-truth Wikipedia IDs, popularity scores, and decile labels.

In [3]:
print("Loading questions...")
qa_df = pd.read_parquet(QUESTIONS_PATH)
qa_df = qa_df.dropna(subset=["question_text"])
qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(str).str.strip()

if MAX_QUESTIONS:
    qa_df = qa_df.sample(n=min(MAX_QUESTIONS, len(qa_df)), random_state=42)
    print(f"  Limited to {len(qa_df):,} questions for testing")

print(f"✓ Loaded {len(qa_df):,} questions")
print(f"  Unique docs: {qa_df['wikipedia_id'].nunique():,}")
if "dataset" in qa_df.columns:
    print(f"  Datasets: {qa_df['dataset'].value_counts().to_dict()}")

print("\nSample questions:")
display(qa_df.head(3))

Loading questions...
✓ Loaded 50,575 questions
  Unique docs: 44,857
  Datasets: {'trex': 13080, 'pop_qa': 11125, 'hotpot_qa': 9842, 'natural_questions': 6432, 'trivia_qa': 5465, 'fever': 4631}

Sample questions:


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile
0,3775083,Who was the producer of Competition?,[American Film Manufacturing Company],14902716,Competition (1915 film),18.1250,4.461622e+06,pop_qa,2,0,0
1,5920612,Who was the director of The Day?,[Alfred Rolfe],32987749,The Day (1914 film),19.5625,4.337318e+06,pop_qa,2,0,0
2,488360,What is Carlos María Ramírez's occupation?,[journalist],37748489,Carlos María Ramírez,19.6250,4.353555e+06,pop_qa,2,0,0


## 3. Load FAISS Index

Initialise `FaissRagService` and load the pre-built index from disk.  
The embedding model **must** match what was used during indexing / migration.

In [4]:
print("Loading FAISS index...")

config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    gpu_batch_size=GPU_BATCH_SIZE,
    request_batch_size=REQUEST_BATCH_SIZE,
    normalise_embeddings=NORMALISE_EMBEDDINGS,
    trust_remote_code=True,
    use_progress=False,
)

service = FaissRagService(
    config=config,
    strategy=FAISS_STRATEGY,
    distance_strategy="cosine",
    memory_config=MemoryConfig(use_mmap=USE_MMAP),
)

service.load_index(FAISS_INDEX_DIR, use_mmap=USE_MMAP)

# Set nprobe on the underlying FAISS index for higher recall at query time
import faiss as _faiss
_faiss.ParameterSpace().set_index_parameter(service._faiss_store.index, "nprobe", IVFPQ_NPROBE)

stats = service.get_index_stats()
print(f"\n✓ Index loaded:")
for k, v in stats.items():
    print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

Loading FAISS index...

✓ Index loaded:
  loaded: 1
  is_mmap: 1
  n_vectors: 24,806,534
  strategy: ivfpq
  nlist: 4,096
  nprobe: 512


## 4. Run Batch Retrieval

For each strategy, retrieve the top-K documents for every question.  
Results are skipped if a parquet file from a previous run already exists.

Each result row contains:
- `topk_ids`: Wikipedia IDs of retrieved documents
- `topk_scores`: Relevance scores
- `topk_popularities`: Popularity values of retrieved documents

In [5]:
print("Running retrieval for all strategies...\n")

retrieve_k = max(TOP_K, max(K_VALUES_DETAILED))
results_by_strategy = {}

for strategy in STRATEGIES:
    print(f"{'=' * 60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'=' * 60}")

    output_path = RESULTS_DIR / f"results_{NAME}_{strategy}.parquet"
    if output_path.exists():
        print(f"⏭  Skipping {strategy} — results already exist at {output_path}\n")
        results_by_strategy[strategy] = pd.read_parquet(output_path)
        continue

    # batch_retrieve_with_scores returns list[list[tuple[Document, float]]]
    all_results = service.batch_retrieve_with_scores(
        queries=qa_df["question_text"].tolist(),
        top_k=retrieve_k,
        progress_bar=True,
    )

    # Build results DataFrame (schema matches rag_retrieval.ipynb / eval notebooks)
    rows = []
    for question_row, retrieved_docs in zip(qa_df.itertuples(), all_results):
        expected_id = str(question_row.wikipedia_id).strip()

        retrieved_ids = []
        retrieved_scores = []
        retrieved_popularities = []

        for doc, score in retrieved_docs:
            raw_id = doc.metadata.get("wikipedia_id", doc.metadata.get("id", ""))
            doc_id = str(int(float(raw_id))) if raw_id not in (None, "") else ""
            retrieved_ids.append(doc_id)
            retrieved_scores.append(float(score))
            retrieved_popularities.append(doc.metadata.get("popularity_avg"))

        rows.append({
            "question":                  question_row.question_text,
            "wikipedia_id":              expected_id,
            "wikipedia_title":           getattr(question_row, "wikipedia_title", None),
            "popularity_avg":            getattr(question_row, "popularity_avg", None),
            "dataset":                   getattr(question_row, "dataset", None),
            COL_DECILE_UNWEIGHTED:       getattr(question_row, COL_DECILE_UNWEIGHTED, -1),
            COL_DECILE_CHUNK_WEIGHTED:   getattr(question_row, COL_DECILE_CHUNK_WEIGHTED, -1),
            "decile":                    getattr(question_row, "decile", -1),
            "topk_ids":                  retrieved_ids,
            "topk_scores":               retrieved_scores,
            "topk_popularities":         retrieved_popularities,
        })

    results_df = pd.DataFrame(rows)
    results_by_strategy[strategy] = results_df

    results_df.to_parquet(output_path)
    print(f"  💾 Saved {strategy} → {output_path} ({len(results_df):,} rows)\n")

ALL_STRATEGIES = list(results_by_strategy.keys())
print(f"✅ All retrieval complete! Tested {len(ALL_STRATEGIES)} configurations:")
print(f"   {', '.join(ALL_STRATEGIES)}")

Running retrieval for all strategies...

Strategy: IVFPQ


Retrieving (ivfpq): 100%|██████████| 50575/50575 [10:16:44<00:00,  1.37it/s]   


  💾 Saved ivfpq → /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k/results_ivfpq_high_ivfpq.parquet (50,575 rows)

✅ All retrieval complete! Tested 1 configurations:
   ivfpq


## 5. Save Results

Save retrieval results as Parquet files — one per strategy.  
Results are already saved inside the retrieval loop above; this cell re-saves any
strategy that was loaded from a previous run (idempotent).

In [6]:
print("Saving retrieval results...\n")

for strategy in ALL_STRATEGIES:
    results_df = results_by_strategy[strategy]
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    results_df.to_parquet(output_path)
    print(f"  ✓ Saved {strategy}: {output_path} ({len(results_df):,} rows)")

print(f"\n✅ All results saved to: {RESULTS_DIR}")
print(f"\nNext step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.")

Saving retrieval results...

  ✓ Saved ivfpq: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k/results_ivfpq.parquet (50,575 rows)

✅ All results saved to: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k

Next step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.


## 6. Generate Metadata

Calculate decile boundaries (both unweighted and chunk-weighted) and save configuration metadata.  
This metadata is used by the eval notebooks for per-decile analysis.

In [7]:
from src.metrics.decile_utils import (
    compute_corpus_boundaries,
    boundaries_to_metadata,
    print_boundaries,
)

metadata_path = RESULTS_DIR / "metadata.json"

if metadata_path.exists():
    print(f"⏭  Skipping — metadata already exists at {metadata_path}")
    with open(metadata_path) as f:
        metadata = json.load(f)
    print(f"   Collection: {metadata.get('collection_name', '?')}")
    print(f"   Corpus found: {metadata.get('corpus_found', '?')}")
else:
    print("Calculating decile boundaries from corpus...\n")

    CORPUS_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"

    if not CORPUS_PATH.exists():
        print(f"⚠️  Corpus not found at {CORPUS_PATH}")
        print("   Metadata will be saved without decile boundaries")
        metadata = {
            "collection_name": COLLECTION_NAME,
            "output_name": OUTPUT_NAME,
            "embedding_model": EMBEDDING_MODEL,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "strategies": ALL_STRATEGIES,
            "top_k": TOP_K,
            "k_values_detailed": K_VALUES_DETAILED,
            "ivfpq_nprobe": IVFPQ_NPROBE,
            "num_questions": len(qa_df),
            "corpus_path": str(CORPUS_PATH),
            "corpus_found": False,
        }
    else:
        print(f"Reading corpus: {CORPUS_PATH}")

        boundaries_uw, boundaries_cw, stats, _ = compute_corpus_boundaries(
            corpus_path=CORPUS_PATH,
            batch_size=100_000,
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )

        print(f"\n✓ Calculated decile boundaries")
        print(f"  Unique documents: {stats['unique_documents_with_popularity']:,}")
        print(f"  Total chunks: {stats['total_chunks_after_splitting']:,}")
        print_boundaries(boundaries_uw, boundaries_cw)

        metadata = {
            "collection_name": COLLECTION_NAME,
            "output_name": OUTPUT_NAME,
            "embedding_model": EMBEDDING_MODEL,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "strategies": ALL_STRATEGIES,
            "top_k": TOP_K,
            "k_values_detailed": K_VALUES_DETAILED,
            "ivfpq_nprobe": IVFPQ_NPROBE,
            "num_questions": len(qa_df),
            "corpus_path": str(CORPUS_PATH),
            "corpus_found": True,
            **boundaries_to_metadata(boundaries_uw, boundaries_cw, stats, CHUNK_SIZE, CHUNK_OVERLAP),
        }

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\n✅ Metadata saved to: {metadata_path}")
    print(f"\nReady for evaluation! Open rag_evaluation.ipynb to analyze results.")

⏭  Skipping — metadata already exists at /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k/metadata.json
   Collection: wiki_full_l
   Corpus found: True
